# Lab Tasks - Solutions

In this notebook we will use linear regression to analyse the *2024 World Happiness Index* data, compiled by Gallup [here](https://data.worldhappiness.report/). This dataset provides country-level measurements of various social and economic factors that potentially influence national well-being.

Each row represents a country with the following features:

- *country*: name of the country
- *gdp*: contribution to happiness from economic output per person (log scaled)
- *social_support*: contribution from having someone to count on in times of trouble
- *health*: contribution from healthy life expectancy (in years)
- *freedom*: contribution from perceived freedom to make life choices (autonomy)
- *generosity*: contribution from perceived level of generosity of citizens (charitable giving behaviour)
- *corruption*: contribution to happiness from trust in government and business (lower corruption = higher trust)

## Task 1

Use Python to download the World Happiness Index data from the link below and load it into a Pandas DataFrame. 

http://mlg.ucd.ie/modules/python/happiness2024.csv

In [ ]:
import pandas as pd
# Pandas can download the data directly from the URL
df = pd.read_csv("http://mlg.ucd.ie/modules/python/happiness2024.csv", index_col="country")
# check the basic structure
print(f"DataFrame has {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
# display the first 10 rows
df.head(10)

Calculate basic summary statistics for the data and list the top 5 ranked countries for each measure.

In [ ]:
# get basic statistics for the various measures (columns)
display(df.describe())

# list the top 5 ranked countries for each measure (column)
for column in df.columns:
	print(f"Top 5 countries for {column}:")
	display(df[column].sort_values(ascending=False).head(5).to_frame())

## Task 2

Calculate the correlations between the different measures in the data to help us understand which factors tend to co-occur.

In [ ]:
# calculate the pairwise correlations
df_c = df.corr()
display(df_c)

# we can turn this into a sorted DataFrame to show ranking for column pairs
# with the highest and lowest correlation
from itertools import combinations
rows = []
for v1, v2 in combinations(df_c.columns, 2):
    rows.append({"Measure 1": v1, "Measure 2": v2, "Correlation": df_c[v1][v2]})
# show the ranked list
pd.DataFrame(rows).sort_values(by="Correlation", ascending=False)

## Task 3

Apply **simple linear regression** to model the relationship where *gdp* is the input variable (predictor) and *health* is the target variable (response). 

Create a scatter plot with the regression line overlaid to visualise the relationship.

In [ ]:
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
%matplotlib inline

# prepare data for regression analysis
x = df[["gdp"]].values
y = df[["health"]].values

# build and fit the regression model
model = LinearRegression()
model.fit(x, y)
print(f"Coefficient: {model.coef_[0][0]:.4f}")
print(f"Intercept: {model.intercept_[0]:.4f}")

# visualise the relationship with regression line
plt.figure(figsize=(8, 5.5))
plt.scatter(x, y, alpha=0.7)

# plot the fitted regression line
m = model.coef_[0]
b = model.intercept_
x_min = x.min()
x_max = x.max()
# this draws the regression line from x_min to x_max using the fitted slope and intercept
plt.plot([x_min, x_max], [m * x_min + b, m * x_max + b], "r", linewidth=2)

plt.xlabel("GDP per Capita", fontsize=12)
plt.ylabel("Health (Life Expectancy)", fontsize=12)
plt.title("GDP vs Health: Linear Regression")
plt.show()

## Task 4

Repeat the **simple linear regression** process from Task 4, but this time use *generosity* as the target (response) variable. 

Visually compare the strength of the relationships between the two regression models from Tasks 4 and 5.

In [ ]:
# analyse gdp vs generosity relationship
x = df[["gdp"]].values
y = df[["generosity"]].values

# build and fit the regression model
model = LinearRegression()
model.fit(x, y)
print(f"Coefficient: {model.coef_[0][0]:.4f}")
print(f"Intercept: {model.intercept_[0]:.4f}")

# visualise the relationship
plt.figure(figsize=(8, 5.5))
plt.scatter(x, y, alpha=0.7)

# plot the fitted regression line
m = model.coef_[0]
b = model.intercept_
x_min = x.min()
x_max = x.max()
# this draws the regression line from x_min to x_max using the fitted slope and intercept
plt.plot([x_min, x_max], [m * x_min + b, m * x_max + b], "r", linewidth=2)

plt.xlabel("GDP per Capita", fontsize=12)
plt.ylabel("Generosity", fontsize=12)
plt.title("GDP vs Generosity: Linear Regression", fontsize=12)
plt.show()

## Task 5

We will now use **multiple linear regression** to model the relationship between several socioeconomic measures and health outcomes.

Firstly, select *gdp*, *social_support*, *freedom*, and *corruption* as the input (predictor) variables, with *health* as the target variable. 

Next, apply **Z-score normalisation** to the input data. This allows the direct comparison of coefficient magnitudes.

Finally, build and fit a multiple linear regression model. Compare the resulting model's coefficients to determine which factor has the strongest relationship with health.

Hint: We can earily apply Z-score normalisation with scikit-learn using `sklearn.preprocessing.StandardScaler`

In [ ]:
from sklearn.preprocessing import StandardScaler

# prepare the data for multiple regression
predictors = ["gdp", "social_support", "freedom", "corruption"]
X = df[predictors].values
y = df[["health"]].values

# apply normalisation to allow direct comparison of coefficients
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# build and fit the multiple regression model on the normalised data
model = LinearRegression()
model.fit(X_scaled, y)

# report the coeffecients for each predictor
# note if the features in the data are strongly correlated, coefficients can be unreliable 
# and may not show true feature importance
print("Coefficients:")
for predictor, coef in zip(predictors, model.coef_[0]):
	print(f"- {predictor}: {coef:.4f}")